In [2]:
import Pkg; Pkg.activate(@__DIR__); Pkg.instantiate()

  Activating project at `~/.julia/dev/drag-formation`


In [3]:
using LinearAlgebra
using SparseArrays

In [4]:
### Constants
μ_mks = 398600.44 #Earth standard gravitational parameter (km^3/sec^2)
J2 = 0.00108262668 #Earth J2 gravity coefficient (dimensionless)
Re_mks = 6378.137 #Radius of the Earth (km)
Ωe_mks = 7.292115e-5 #Rotation rate of Earth (rad/sec)

Cd = 2.2 #Drag coefficient (unitless)
AoM_mks = (2+sqrt(2))*3e-4*1e-4/3.0; #Area-to-mass ratio (km^2/kg) assumes a 3U CubeSat with deployable panels and mass of 3 kg
AoM0_mks = 1e-4*1e-4/3.0; #Area-to-mass ratio (km^2/kg) assumes a 3U CubeSat with deployable panels and mass of 3 kg

In [5]:
#Orbital Parameters (roughly ISS)
a_mks = 6791.0 #semi-major axis (km)
inc = 51.5 #inclination (deg)
n_mks = sqrt(μ_mks/a_mks^3) #mean motion (rad/sec)
P_mks = 2*pi/n_mks; #orbital period (sec)

In [6]:
#Normalize units
a = 1.0 #distance unit is semi-major axis
P = 1.0 #time unit is orbital period
n = 2*pi
μ = (2*pi)^2 #μ_mks*(P_mks^2/a_mks^3)
Re = Re_mks/a_mks
Ωe = Ωe_mks*P_mks;

#a0 = a0_mks/a_mks
#H = H_mks/a_mks
#ρ0 = ρ0_mks*(a_mks^3)
AoM = AoM_mks/(a_mks^2)
AoM0 = AoM0_mks/(a_mks^2);

In [7]:
### Coordinate and unit transformations

function cartesian_to_spherical(x)
    r = sqrt(x[1:3]'*x[1:3]) #range
    ϕ = atan(x[2],x[1]) #azimuth
    θ = atan(x[3],sqrt(x[1:2]'*x[1:2])) #elevation measured up from equatorial plane
    
    return [r; ϕ; θ]
end

function spherical_to_cartesian(s)
    r = s[1] #range
    ϕ = s[2] #azimuth
    θ = s[3] #elevation measured up from equatorial plane
    
    x = r*cos(θ)*cos(ϕ)
    y = r*cos(θ)*sin(ϕ)
    z = r*sin(θ)
    
    return [x; y; z]
end

function rtn_frame(x0)
    r = x0[1:3]
    v = x0[4:6]
    n = cross(r,v)
    
    R = r/norm(r)
    N = n/norm(n)
    T = cross(N,R)
    
    return [R T N] #rotation matrix from RTN to Cartesian frame
end

function rtn_to_cartesian(x0,δr)
    r0 = x0[1:3]
    v0 = x0[4:6]
    
    R = rtn_frame(x0) #rotation from RTN to Cartesian frame
    r = R*δr + r0 #position in inertial frame
    
    return r
end

function cartesian_to_rtn(x0,r)
    r0 = x0[1:3]
    v0 = x0[4:6]
    
    R = rtn_frame(x0) #rotation from RTN to Cartesian frame
    δr = R'*(r - r0) #position in RTN frame
    
    return δr
end

function cartesian_to_ΔaθΩ(x0,x1)
    r0 = x0[1:3]
    v0 = x0[4:6]
    
    r1 = x1[1:3]
    v1 = x1[4:6]
    
    E0 = 0.5*v0'*v0 + potential(cartesian_to_spherical(x0))
    E1 = 0.5*v1'*v1 + potential(cartesian_to_spherical(x1))
    
    a0 = -μ/(2*E0)
    a1 = -μ/(2*E1)
    Δa = a1-a0
    
    R = rtn_frame(x0) #rotation from RTN to Cartesian frame    
    rtn1 = R'*r1
    Δθ = atan(rtn1[2]/norm(rtn1[1:2]), rtn1[1]/norm(rtn1[1:2]))
    
    h0 = cross(r0,v0)
    Ω0 = atan(h0[2]/norm(h0[1:2]), h0[1]/norm(h0[1:2]))
    
    h1 = cross(r1,v1)
    Ω1 = atan(h1[2]/norm(h1[1:2]), h1[1]/norm(h0[1:2]))
    
    ΔΩ = Ω1-Ω0
    
    return [Δa; Δθ; ΔΩ]
end

function mks_to_orbital(x)
    #change state from MKS units to orbital units
    
    return [x[1:3]/a_mks; x[4:6]*(P_mks/a_mks)]
end

function orbital_to_mks(x)
    #change state from orbital units to MKS
    
    return [x[1:3]*a_mks; x[4:6]*(a_mks/P_mks)]
end

orbital_to_mks (generic function with 1 method)

In [8]:
function density_mks(alt)
    #This fit is good from 250-500 km altitude
    #Units are km, kg
    a = 106.2
    b = -0.03243
    c = 1.988
    d = -0.01672
    ρ = a*exp(b*alt) + c*exp(d*alt)
end

function atm_density(r)
    #ρ = ρ0*exp((a0-r)/H)
    ρ = (a_mks^3)*density_mks(r*a_mks - Re_mks)
end

atm_density (generic function with 1 method)

In [9]:
#Drag perturbation at initial altitude

#Fd_mks = 0.5*Cd*density_mks(a_mks-Re_mks)*(AoM_mks)*(n_mks*a_mks)^2
#ȧ_mks = 2*(a_mks^2/μ_mks)*a_mks*n_mks*Fd_mks

#a_mid = a-5.0/a_mks #evaluate drag at average altitude of maneuver
Fd = -0.5*Cd*atm_density(a)*(AoM)*(n*a)^2 #0.965*
ȧ = 2*(a^2/μ)*(a)*n*Fd

-6.922788074332244e-6

In [10]:
#Continuous-Time Linear model
#state is [Δθ; Δa; ΔΩ] where these are differences between satellite 1 and satellite 2
#Can treat drag input as positive part going to sat 2 and negative part going to sat 1
# delta is always with respect to sat 1

k1 = (21.0/4.0)*J2*Re*Re*sqrt(μ/a^9)*cosd(inc) #dΩ̇/da
k2 = -(3.0/2)*sqrt(μ/a^5) #dθ̇/da (dn/da) #(21.0/4.0)*J2*Re*Re*sqrt(μ/a^9)*(1-1.5*(sind(inc))^2)#
k3 = ȧ #dȧ/d(Drag)

A0 = [0 k2 0; 0 0 0; 0 k1 0]
B0 = [0; k3; 0]





3-element Vector{Float64}:
  0.0
 -6.922788074332244e-6
  0.0

In [11]:
#Discrete-time model
num_sat = 2 # number of couple of satellites 

A_bar = zeros(3*num_sat,3*num_sat) 
B_bar = zeros(3*num_sat,num_sat) 

for k =0:num_sat-1
    A_bar[1+(3*k):3+(3*k),1+(3*k):3+(3*k)] = A0
    B_bar[1+(3*k):3+(3*k),k+1] = B0
#     B_bar[1+(3*k):3+(3*k),1] = B0
end

h = 0.5 #time step
A = I + h*A_bar
B = h*B_bar + 0.5*h*h*A_bar*B_bar;

B_bar

6×2 Matrix{Float64}:
  0.0          0.0
 -6.92279e-6   0.0
  0.0          0.0
  0.0          0.0
  0.0         -6.92279e-6
  0.0          0.0

In [12]:
#Initial and terminal constraints
l_12 = 1.0 # number of revolutions
l_23 = 1.0
Δx0 = zeros(size(A,1))
Δθf = 2.0*pi #one full revolution (can be any multiple of 2π)
Δaf = 0; #end at the same altitude

# Right Ascension Separation
k4 = k1/k2
ΔΩf = k4*(Δθf)

-0.013073637274010471

In [13]:
#Inequality constraints
u_min = -1
u_max = 1
Δa_min = -10/a_mks
Δa_max = 10/a_mks

0.0014725371815638344

In [14]:
#Discretized Optimal Control Problem Parameters
Nx = size(A,1)     # number of state
Nu = size(B,2)     # number of controls
Tfinal = 1500 # final time (number of orbits)
Nt = Int(Tfinal/h)+1    # number of time steps
Nz = Nt*(Nx+Nu) #number of decision variables
thist = Array(range(0,h*(Nt-1), step=h));


In [1]:
#Useful Matrices

# decision variables are stacked z = [u1; x2; u2; x3; u3; ...; xf]

U_1 = kron(Diagonal(I,Nt),  [1.0 0 0 0 0 0 0 0]) #Matrix that picks out all u
U_2 = kron(Diagonal(I,Nt),  [0 1.0 0 0 0 0 0 0]) #Matrix that picks out all u
X1_1 = kron(Diagonal(I,Nt), [0 0 1.0 0 0 0 0 0]) #Matrix that picks out all x1 = Δθ
X2_1 = kron(Diagonal(I,Nt), [0 0 0 1.0 0 0 0 0]) #Matrix that picks out all x2 = Δa
X3_1 = kron(Diagonal(I,Nt), [0 0 0 0 1.0 0 0 0]) #Matrix that picks out all x3 = ΔΘ

X1_2 = kron(Diagonal(I,Nt), [0 0 0 0 0 1.0 0 0]) #Matrix that picks out all x1 = Δθ
X2_2 = kron(Diagonal(I,Nt), [0 0 0 0 0 0 1.0 0]) #Matrix that picks out all x2 = Δa
X3_2 = kron(Diagonal(I,Nt), [0 0 0 0 0 0 0 1.0]) #Matrix that picks out all x3 = ΔΘ

X1F_1 = zeros(1,(Nu+Nx)*Nt); X1F_1[end-5] = 1.0 #Matrix that picks out x1f = Δθf
X2F_1 = zeros(1,(Nu+Nx)*Nt); X2F_1[end-4] = 1.0 #Matrix that picks out x2f = Δaf
X3F_1 = zeros(1,(Nu+Nx)*Nt); X3F_1[end-3] = 1.0 #Matrix that picks out x3f = ΔΘf
X1F_2 = zeros(1,(Nu+Nx)*Nt); X1F_1[end-2] = 1.0 #Matrix that picks out x1f = Δθf
X2F_2 = zeros(1,(Nu+Nx)*Nt); X2F_1[end-1] = 1.0 #Matrix that picks out x2f = Δaf
X3F_2 = zeros(1,(Nu+Nx)*Nt); X3F_1[end] = 1.0 #Matrix that picks out x3f = ΔΘf

# # 1 control inputs 
# U = kron(Diagonal(I,Nt), [I zeros(1,3*num_sat)]) #Matrix that picks out all u
# X1 = kron(Diagonal(I,Nt), repeat([zeros(1,1) I zeros(1,1)],1,num_sat)) #Matrix that picks out all x1 = Δθ
# X2 = kron(Diagonal(I,Nt), [zeros(1,1) repeat([zeros(1,1) I zeros(1,1)],1,num_sat)]) #Matrix that picks out all x2 = Δa
# X3 = kron(Diagonal(I,Nt), [zeros(1,1) repeat([zeros(1,2) I],1,num_sat)]) #Matrix that picks out all x3 = ΔΘ
# X1F = zeros(1,(Nu+Nx)*Nt); X1F[end-(Nu+Nx)+2:3:end] .= 1.0; #Matrix that picks out x1f = Δθf
# X2F = zeros(1,(Nu+Nx)*Nt); X2F[end-(Nu+Nx)+3:3:end] .= 1.0; #Matrix that picks out x2f = Δaf
# X3F = zeros(1,(Nu+Nx)*Nt); X3F[end-(Nu+Nx)+4:3:end] .= 1.0;#Matrix that picks out x3f = ΔΘf


# # 2 control inputs 
# U = kron(Diagonal(I,Nt), repeat([I zeros(1,3)],1,num_sat)) #Matrix that picks out all u
# X1 = kron(Diagonal(I,Nt), repeat([zeros(1,1) I zeros(1,2)],1,num_sat)) #Matrix that picks out all x1 = Δθ
# X2 = kron(Diagonal(I,Nt), repeat([zeros(1,2) I zeros(1,1)],1,num_sat)) #Matrix that picks out all x2 = Δa
# X3 = kron(Diagonal(I,Nt), repeat([zeros(1,3) I],1,num_sat)) #Matrix that picks out all x3 = ΔΘ
# X1F = zeros(1,(Nu+Nx)*Nt); X1F[end-(Nu+Nx)+2:4:end] .= 1.0; #Matrix that picks out x1f = Δθf
# X2F = zeros(1,(Nu+Nx)*Nt); X2F[end-(Nu+Nx)+3:4:end] .= 1.0; #Matrix that picks out x2f = Δaf
# X3F = zeros(1,(Nu+Nx)*Nt); X3F[end-(Nu+Nx)+4:4:end] .= 1.0;#Matrix that picks out x3f = ΔΘf

#Dynamics equality constraints C*z = d
C = sparse([[B -I zeros(Nx,(Nt-1)*(Nu+Nx))]; zeros(Nx*(Nt-1),Nu) [kron(Diagonal(I,Nt-1), [A B]) zeros((Nt-1)*Nx,Nx)] + [zeros((Nt-1)*Nx,Nx) kron(Diagonal(I,Nt-1),[zeros(Nx,Nu) Diagonal(-I,Nx)])]])
d = [-A*Δx0; zeros(Nx*(Nt-1))];

U_2

LoadError: UndefVarError: Diagonal not defined

In [ ]:
#Convex Formulation
using Convex

z = Variable(Nz)

problem = minimize(sum(abs(X1_1*z - Δθf)) + sum(abs(U_1*z))+ sum(abs(X1_2*z - Δθf)) + sum(abs(U_2*z)));  #+ sum(abs(X1_2*z - Δθf)) + sum(abs(U_2*z))
problem.constraints += [C*z == d] #dynamics constraints
# problem.constraints += [X1F_1*z == Δθf] #final true anomaly constraint
problem.constraints += [X2F_1*z == Δaf] #final altitude constraint
problem.constraints += [X2F_2*z == Δaf] #final altitude constraint
# problem.constraints += [X3F_1*z == ΔΩf] #final true anomaly constraint
problem.constraints += [U_1*z >= u_min; U_1*z <= u_max] #drag constraints
problem.constraints += [X2_1*z >= Δa_min; X2_1*z <= Δa_max]; #altitude constraints
problem.constraints += [U_2*z >= u_min; U_2*z <= u_max] #drag constraints
problem.constraints += [X2_2*z >= Δa_min; X2_2*z <= Δa_max]; #altitude constraints

In [ ]:
#using ECOS
#solve!(problem, MOI.OptimizerWithAttributes(ECOS.Optimizer, "maxit" => 10000))

using MosekTools
solve!(problem, Mosek.Optimizer)

In [ ]:
# U_1 = kron(Diagonal(I,Nt),  [1.0 0 0 0 0 0 0]) #Matrix that picks out all u
# X1_1 = kron(Diagonal(I,Nt), [0 1.0 0 0 0 0 0]) #Matrix that picks out all x1 = Δθ
# X2_1 = kron(Diagonal(I,Nt), [0 0 1.0 0 0 0 0]) #Matrix that picks out all x2 = Δa
# X3_1 = kron(Diagonal(I,Nt), [0 0 0 1.0 0 0 0]) #Matrix that picks out all x3 = ΔΘ
# # U_2 = kron(Diagonal(I,Nt),  [0 0 0 0 1.0 0 0 0]) #Matrix that picks out all u
# X1_2 = kron(Diagonal(I,Nt), [0 0 0 0 1.0 0 0]) #Matrix that picks out all x1 = Δθ
# X2_2 = kron(Diagonal(I,Nt), [0 0 0 0 0 1.0 0]) #Matrix that picks out all x2 = Δa
# X3_2 = kron(Diagonal(I,Nt), [0 0 0 0 0 0 1.0]) #Matrix that picks out all x3 = ΔΘ


u1hist = U_1*z.value;
θ1hist = [Δx0[1]; X1_1*z.value];
a1hist = [Δx0[2]; X2_1*z.value];
Ω1hist = [Δx0[3]; X3_1*z.value];

u2hist = U_2*z.value;
θ2hist = [Δx0[1]; X1_2*z.value];
a2hist = [Δx0[2]; X2_2*z.value];
Ω2hist = [Δx0[3]; X3_2*z.value];


# # θ2hist = [Δx0[4]; X1*z.value];
# # a2hist = [Δx0[5]; X2*z.value];
# # Ω2hist = [Δx0[6]; X3*z.value];
# ures = zeros(Nt,Nu)
# println(length(z.value))
# println(Nt)
# println(length(uhist))
# println(length(U))
# for i = 1:num_sat
#     temp = uhist[i:num_sat:end]
#     println(temp)
#     print(ures[:,i])
# end

In [ ]:
using Plots
plotlyjs()
plot(thist,u1hist)
plot!(thist,u2hist)

In [ ]:
plot(thist[1:end],θ1hist[2:end,1])
plot!(thist[:],θ2hist[2:end,1])

In [ ]:
plot(thist,a_mks*a1hist[2:end,1])
plot!(thist,a_mks*a2hist[2:end,1])

In [ ]:
plot(thist,Ω1hist[2:end,1])
plot!(thist,Ω2hist[2:end,1])

In [ ]:
### Full Orbital Dynamics with J2 and Drag

using ForwardDiff

function potential(s)
    #Input is position in spherical coordinates
    r = s[1] #range
    #ϕ = s[2] #azimuth
    θ = s[3] #elevation measured up from equatorial plane
    
#     U = -(μ/r) #spherical
    U = -(μ/r) - (0.5*J2*μ*Re*Re/r^3)*(3*(sin(θ)^2) - 1.0) #with J2
end

function potential_noJ2(s)
    #Input is position in spherical coordinates
    r = s[1] #range
    #ϕ = s[2] #azimuth
    θ = s[3] #elevation measured up from equatorial plane
    
    U = -(μ/r) #spherical
#     U = -(μ/r) - (0.5*J2*μ*Re*Re/r^3)*(3*(sin(θ)^2) - 1.0) #with J2
end

function satellite_dynamics(x,u)
    #Newton's 2nd law
    
    r = x[1:3]
    v = x[4:6]
    
    J = ForwardDiff.jacobian(cartesian_to_spherical, r)
    g = -ForwardDiff.gradient(potential, cartesian_to_spherical(r))
    
    v_rel = v #- cross([0;0;Ωe],r) #Assume atmosphere co-rotates with the Earth
    ρ = atm_density(norm(r))
    Fdrag = -0.5*(u*AoM)*Cd*ρ*norm(v_rel)*v_rel
    
    v̇ = J'*g + Fdrag
    
    return [v; v̇]
end

function satellite_dynamics_withoutJ2(x,u)
    #Newton's 2nd law
    
    r = x[1:3]
    v = x[4:6]
    
    J = ForwardDiff.jacobian(cartesian_to_spherical, r)
    g = -ForwardDiff.gradient(potential_noJ2, cartesian_to_spherical(r))
    
    v_rel = v #- cross([0;0;Ωe],r) #Assume atmosphere co-rotates with the Earth
    ρ = atm_density(norm(r))
    Fdrag = -0.5*(u*AoM)*Cd*ρ*norm(v_rel)*v_rel
    
    v̇ = J'*g + Fdrag
    
    return [v; v̇]
end

In [ ]:
#Interpolation of control input trajectory from solver

using Interpolations
uinterp = LinearInterpolation(thist, uhist[:]);

function formation_dynamics!(ẋ,x,p,t)    
    x1 = x[1:6]
    x2 = x[7:12]
    
    u = uinterp(t)
    u1 = max(0, -u)
    u2 = max(0, u)
    
    ẋ1 = satellite_dynamics(x1,u1)
    ẋ2 = satellite_dynamics(x2,u2)
    
    ẋ .= [ẋ1; ẋ2]
end

function formation_dynamics_withoutJ2!(ẋ,x,p,t)    
    x1 = x[1:6]
    x2 = x[7:12]
    
    u = uinterp(t)
    u1 = max(0, -u)
    u2 = max(0, u)
    
    ẋ1 = satellite_dynamics_withoutJ2(x1,u1)
    ẋ2 = satellite_dynamics_withoutJ2(x2,u2)
    
    ẋ .= [ẋ1; ẋ2]
end

In [ ]:
#Initial conditions in Cartesian Coordinates
r0 = [a; 0; 0]
v0 = [0; cosd(inc)*sqrt(μ/a); sind(inc)*sqrt(μ/a)]
x0 = [r0; v0];

r1 = [a; 0; 0]
v1 = [0; cosd(inc)*sqrt(μ/(a)); sind(inc)*sqrt(μ/(a))]
x1 = [r1; v1];

In [ ]:
#Full Orbit Mechanics Simulation

using DifferentialEquations

tspan = (0.0, Tfinal*P)

ref_prob = ODEProblem(formation_dynamics!,[x0; x1],tspan)
ref_soln = solve(ref_prob,Tsit5(); abstol=1e-9,reltol=1e-9);

ref_prob_noJ2 = ODEProblem(formation_dynamics_withoutJ2!,[x0; x1],tspan)
ref_soln_noJ2 = solve(ref_prob_noJ2,Tsit5(); abstol=1e-9,reltol=1e-9);

In [ ]:
#Calculate relative positions
Nt = 1000
tsamp = LinRange(tspan[1],tspan[2],Nt)

x0_soln = zeros(6,Nt)
x1_soln = zeros(6,Nt)

x0_noJ2_soln = zeros(6,Nt)
x1_noJ2_soln = zeros(6,Nt)

Δa_soln = zeros(Nt)
Δθ_soln = zeros(Nt)
ΔΩ_soln = zeros(Nt)

a_soln1 = zeros(Nt)
θ_soln1 = zeros(Nt)
Ω_soln1 = zeros(Nt)
u_soln1 = zeros(Nt)

a_soln2 = zeros(Nt)
θ_soln2 = zeros(Nt)
Ω_soln2 = zeros(Nt)
u_soln2 = zeros(Nt)

# tsamp
# uinterp = LinearInterpolation(tsamp, uhist[:])

for k = 1:Nt
    x0_soln[:,k] = ref_soln(tsamp[k])[1:6]
    x1_soln[:,k] = ref_soln(tsamp[k])[7:12]
    x0_noJ2_soln[:,k] = ref_soln_noJ2(tsamp[k])[1:6]
    x1_noJ2_soln[:,k] = ref_soln_noJ2(tsamp[k])[7:12]
    
    aθΩ1 = cartesian_to_spherical(x0_noJ2_soln[:,k])
    aθΩ2 = cartesian_to_spherical(x1_noJ2_soln[:,k])
    ΔaθΩ = cartesian_to_ΔaθΩ(x0_soln[:,k], x1_soln[:,k])
    
    Δa_soln[k] = ΔaθΩ[1]
    Δθ_soln[k] = ΔaθΩ[2]
    ΔΩ_soln[k] = ΔaθΩ[3]
    
    a_soln1[k] = aθΩ1[1]
    θ_soln1[k] = aθΩ1[2]
    Ω_soln1[k] = aθΩ1[3]
    
    a_soln2[k] = aθΩ2[1]
    θ_soln2[k] = aθΩ2[2]
    Ω_soln2[k] = aθΩ2[3]
    u_soln1[k] = max(0, -uinterp(tsamp[k]))
    u_soln2[k] = max(0, uinterp(tsamp[k]))
end
# uinterp
# u_soln2 = max(0, uinterp)

In [ ]:
plot(tsamp,a_mks*Δa_soln)

In [ ]:
plot(tsamp,Δθ_soln)

In [ ]:
plot(tsamp,ΔΩ_soln)

In [ ]:
plot(tsamp,u_soln1,label="Drag Sat 1")
plot!(tsamp,u_soln2,label="Drag Sat 2")

In [ ]:
plot(tsamp,a_mks*a_soln1.-Re_mks,label="Sat 1")
plot!(tsamp,a_mks*a_soln2.-Re_mks,label="Sat 2")
plot!(xlabel= "Time, s",ylabel= "Altitude, km")

In [ ]:
# using NaNStatistics
# window_size = length(tsamp)/1.0
# z1 = movmean(θ_soln1, window_size)
# z2 = movmean(θ_soln2, window_size)
# plot(tsamp,θ_soln1,linestyle=:dot)
# plot!(tsamp,θ_soln2,linestyle=:dot)
# plot!(tsamp,z1, linewidth = 2)
# plot!(tsamp,z2, linewidth = 2)

In [ ]:
# window_size = length(tsamp)/100.0
# z1 = movmean(Ω_soln1, window_size)
# z2 = movmean(Ω_soln2, window_size)
# # plot(tsamp,Ω_soln1,linestyle=:dot)
# # plot!(tsamp,Ω_soln2,linestyle=:dot)
# plot(tsamp,z1, linewidth = 2)
# plot!(tsamp,z2, linewidth = 2)
# plot!(ylim= [-1,1])

In [ ]:
k1/k2

In [ ]:
(k1/k2)*2*pi

In [ ]:
((7.0/2)*(Re*Re/(a*a))*J2*cosd(inc))*2*pi